# Bellabeat — Analyze Phase

Business question: **what trends in smart-device usage could inform Bellabeat's marketing
strategy?**

Uses the cleaned tables produced in the Process phase (`data/processed/`). Small aggregate
tables produced here are saved to `data/summary/` for reuse in the Share phase.

## Setup

In [1]:
import pandas as pd
import os

PROC = "../data/processed"
RAW = "../data/raw"
SUMMARY_DIR = "../data/summary"
os.makedirs(SUMMARY_DIR, exist_ok=True)

activity = pd.read_parquet(os.path.join(PROC, "daily_activity.parquet"))
sleep = pd.read_parquet(os.path.join(PROC, "sleep_day.parquet"))
merged = pd.read_parquet(os.path.join(PROC, "activity_sleep_merged.parquet"))
print(f"Loaded activity: {activity.shape}, sleep: {sleep.shape}, merged: {merged.shape}")

Loaded activity: (1373, 16), sleep: (410, 6), merged: (1373, 21)

## 1. Overall activity — descriptive stats

In [2]:
cols = ["total_steps", "total_distance", "very_active_minutes", "fairly_active_minutes",
        "lightly_active_minutes", "sedentary_minutes", "calories"]
desc = activity[cols].describe().round(1)
desc.to_csv(os.path.join(SUMMARY_DIR, "activity_descriptive_stats.csv"))
desc

       total_steps  total_distance  very_active_minutes  fairly_active_minutes  \
count       1373.0          1373.0                1373.0                 1373.0
mean        7377.4             5.3                  19.9                   13.6
std         5198.1             4.0                  32.9                   20.4
min            0.0             0.0                   0.0                    0.0
25%         3321.0             2.3                   0.0                    0.0
50%         7142.0             5.0                   3.0                    6.0
75%        10645.0             7.6                  30.0                   19.0
max        36019.0            28.0                 210.0                  143.0

       lightly_active_minutes  sedentary_minutes  calories
count                  1373.0             1373.0     1373.0
mean                    188.1             1001.3     2294.8
std                      95.6              304.2      725.5
min                       0.0         

**Average: 7,377 steps/day** — below the commonly-cited 10,000-step benchmark — and **1,001 minutes (16.7h) sedentary per day.**

## 2. Where the tracked day actually goes

In [3]:
intensity_cols = ["very_active_minutes", "fairly_active_minutes", "lightly_active_minutes", "sedentary_minutes"]
mean_minutes = activity[intensity_cols].mean()
total_tracked = mean_minutes.sum()
pct = (mean_minutes / total_tracked * 100).round(1)
intensity_summary = pd.DataFrame({"mean_minutes": mean_minutes.round(1), "pct_of_day": pct})
intensity_summary.to_csv(os.path.join(SUMMARY_DIR, "intensity_breakdown.csv"))
print(intensity_summary)
print(f"\nTotal mean tracked minutes/day: {total_tracked:.1f} ({total_tracked/60:.1f} hours)")

                        mean_minutes  pct_of_day
very_active_minutes            19.9         1.6
fairly_active_minutes          13.6         1.1
lightly_active_minutes        188.1        15.4
sedentary_minutes             1001.3        81.9

Total mean tracked minutes/day: 1222.9 (20.4 hours)

**81.9% of the average tracked day is sedentary** — only 2.7% is very or fairly active combined.

## 3. User segmentation by average daily steps

Using the standard step-count activity bands (CDC/common fitness convention): Sedentary (<5,000),
Lightly active (5,000–7,499), Fairly active (7,500–9,999), Very active (10,000+).

In [4]:
user_avg_steps = activity.groupby("id")["total_steps"].mean()

def classify(steps):
    if steps < 5000:
        return "Sedentary (<5,000)"
    elif steps < 7500:
        return "Lightly active (5,000-7,499)"
    elif steps < 10000:
        return "Fairly active (7,500-9,999)"
    else:
        return "Very active (10,000+)"

segments = user_avg_steps.apply(classify)
seg_counts = segments.value_counts()
seg_pct = (seg_counts / seg_counts.sum() * 100).round(1)
order = ["Sedentary (<5,000)", "Lightly active (5,000-7,499)", "Fairly active (7,500-9,999)", "Very active (10,000+)"]
seg_summary = pd.DataFrame({"n_users": seg_counts, "pct": seg_pct}).reindex(order)
seg_summary.to_csv(os.path.join(SUMMARY_DIR, "user_activity_segments.csv"))
seg_summary

                               n_users   pct
Sedentary (<5,000)                 11  31.4
Lightly active (5,000-7,499)        9  25.7
Fairly active (7,500-9,999)         8  22.9
Very active (10,000+)               7  20.0

**80% of users average fewer than 10,000 steps/day** — most of this sample falls short of the commonly-cited activity benchmark.

## 4. Sleep

In [5]:
sleep_cols = ["total_minutes_asleep", "total_time_in_bed", "sleep_efficiency_pct"]
sleep_desc = sleep[sleep_cols].describe().round(1)
sleep_desc.to_csv(os.path.join(SUMMARY_DIR, "sleep_descriptive_stats.csv"))
print(sleep_desc)
pct_meeting_7h = (sleep["total_minutes_asleep"] >= 420).mean() * 100
print(f"\n% of sleep-nights with >= 7h (420 min) asleep: {pct_meeting_7h:.1f}%")

       total_minutes_asleep  total_time_in_bed  sleep_efficiency_pct
count                  410.0              410.0                  410.0
mean                   419.2              458.5                   91.6
std                    118.6              127.5                    8.7
min                     58.0               61.0                   49.8
25%                    361.0              403.8                   91.2
50%                    432.5              463.0                   94.3
75%                    490.0              526.0                   96.1
max                    796.0              961.0                  100.0

% of sleep-nights with >= 7h (420 min) asleep: 55.9%

Sleep **efficiency** is high (91.6% average — once in bed, users are mostly asleep), but **44% of nights fall short of 7 hours** of total sleep.

## 5. Correlations

In [6]:
corr_steps_sedentary = activity["total_steps"].corr(activity["sedentary_minutes"])
corr_steps_calories = activity["total_steps"].corr(activity["calories"])
corr_sedentary_sleep = merged["sedentary_minutes"].corr(merged["total_minutes_asleep"])
corr_steps_sleep = merged["total_steps"].corr(merged["total_minutes_asleep"])

corr_summary = pd.DataFrame({
    "pair": ["steps vs sedentary_minutes", "steps vs calories",
             "sedentary_minutes vs minutes_asleep", "steps vs minutes_asleep"],
    "correlation": [corr_steps_sedentary, corr_steps_calories, corr_sedentary_sleep, corr_steps_sleep],
}).round(3)
corr_summary.to_csv(os.path.join(SUMMARY_DIR, "correlations.csv"), index=False)
corr_summary

                                   pair  correlation
0            steps vs sedentary_minutes       -0.356
1                    steps vs calories        0.580
2  sedentary_minutes vs minutes_asleep       -0.601
3              steps vs minutes_asleep       -0.190

The strongest relationship in the data is **sedentary minutes vs. sleep (-0.601)**: the more
sedentary time in a user's day, the less they sleep that night — a substantive, non-obvious
finding for a wellness brand. Steps vs. calories is positive as expected (0.580); steps vs. sleep
is only weakly negative (-0.190).

## 6. Hourly activity pattern (when in the day are people active?)

In [7]:
h1 = pd.read_csv(os.path.join(RAW, "2016-03-12_to_2016-04-11", "hourlySteps_merged.csv"), parse_dates=["ActivityHour"])
h2 = pd.read_csv(os.path.join(RAW, "2016-04-12_to_2016-05-12", "hourlySteps_merged.csv"), parse_dates=["ActivityHour"])
hourly = pd.concat([h1, h2], ignore_index=True)
hourly.columns = ["id", "activity_hour", "step_total"]
hourly["hour"] = hourly["activity_hour"].dt.hour
by_hour = hourly.groupby("hour")["step_total"].mean().round(1)
by_hour.to_csv(os.path.join(SUMMARY_DIR, "steps_by_hour.csv"))
print(by_hour)
peak_hour = by_hour.idxmax()
print(f"\nPeak hour: {peak_hour}:00 with avg {by_hour[peak_hour]:.1f} steps")

hour
0      43.4
1      21.7
2      13.7
3       6.9
4      11.2
5      34.6
6     148.1
7     285.8
8     395.3
9     431.7
10    458.5
11    454.7
12    534.3
13    496.2
14    506.0
15    398.1
16    471.0
17    499.7
18    550.3
19    554.9
20    377.6
21    283.5
22    204.0
23    112.1
Name: step_total, dtype: float64

Peak hour: 19:00 with avg 554.9 steps

Activity is low overnight, rises sharply from 6 AM, stays elevated through the day with a
midday bump (12–1 PM) and the **highest average steps in the early evening (6–8 PM)** — no sharp
single spike, a broad active window from mid-morning to evening.

## Summary of key findings

1. **Most users don't hit 10,000 steps/day**: 80% of the sample averages under 10,000 steps
   (31% under 5,000). Average is 7,377 steps/day.
2. **The tracked day is overwhelmingly sedentary**: 81.9% of tracked minutes are sedentary; only
   2.7% are very/fairly active combined.
3. **Sleep is efficient but often insufficient**: 91.6% average sleep efficiency, but 44% of
   nights fall short of 7 hours of sleep.
4. **Sedentary time is the strongest predictor of poor sleep** in this data (r = -0.601) — a much
   stronger relationship than steps vs. sleep (r = -0.190).
5. **Activity peaks in the early evening** (6–8 PM), with a broad active window from mid-morning
   onward and very little overnight activity.

**Overall**: this sample of smart-device users is largely under-active relative to common
benchmarks, and — more actionably — their sedentary behavior tracks more closely with poor sleep
than their step count does. This reframes the opportunity from "get people to walk more" to
"help people break up sedentary time," which directly informs the Share and Act phases.